***1) PREPROCESSING AND FILTERING***

In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "01A_GSE135779_CHILD_PREPROCESSING")


In [ ]:
#CREATE INDIVIDUAL ADATA FILES

import os
import glob
import gzip
import pandas as pd
import scanpy as sc
from scipy.io import mmread

# -------------------------------
# paths
# -------------------------------
raw_dir = f"{BASE_DIR}/GSE135779_RAW"
out_dir = f"{BASE_DIR}/child_individual_h5ad"
os.makedirs(out_dir, exist_ok=True)

# -------------------------------
# child GSM IDs
# -------------------------------
child_gsms = [
    "GSM4029907", "GSM4029914", "GSM4029915", "GSM4029922", "GSM4029923",
    "GSM4029930", "GSM4029931", "GSM4029936", "GSM4029937", "GSM4029938",
    "GSM4029939", "GSM4029896", "GSM4029897", "GSM4029898", "GSM4029899",
    "GSM4029900", "GSM4029901", "GSM4029902", "GSM4029903", "GSM4029904",
    "GSM4029905", "GSM4029906", "GSM4029908", "GSM4029909", "GSM4029910",
    "GSM4029911", "GSM4029912", "GSM4029913", "GSM4029916", "GSM4029917",
    "GSM4029918", "GSM4029919", "GSM4029920", "GSM4029921", "GSM4029924",
    "GSM4029925", "GSM4029926", "GSM4029927", "GSM4029928", "GSM4029929",
    "GSM4029932", "GSM4029933", "GSM4029934", "GSM4029935"
]

# -------------------------------
# helper functions
# -------------------------------
def find_one_file(patterns):
    """Return first existing match across a list of glob patterns."""
    for pattern in patterns:
        hits = glob.glob(pattern)
        if hits:
            return hits[0]
    return None

def read_table_auto(path, header=None):
    """Read .tsv or .tsv.gz."""
    return pd.read_csv(path, sep="\t", header=header)

def make_unique(names):
    """Simple replacement for older pandas ParserBase dedup."""
    seen = {}
    out = []
    for x in names:
        x = "NA" if pd.isna(x) or str(x).strip() == "" else str(x)
        if x not in seen:
            seen[x] = 0
            out.append(x)
        else:
            seen[x] += 1
            out.append(f"{x}-{seen[x]}")
    return pd.Index(out)

# -------------------------------
# load shared genes file
# -------------------------------
genes_path = os.path.join(raw_dir, "GSE135779_genes.tsv.gz")
genes = read_table_auto(genes_path, header=None)

print("Genes file shape:", genes.shape)

# Common formats:
# col0 = Ensembl/gene_id
# col1 = gene_symbol
if genes.shape[1] >= 2:
    gene_ids = genes.iloc[:, 0].astype(str).values
    gene_symbols = genes.iloc[:, 1].astype(str).values
else:
    gene_ids = genes.iloc[:, 0].astype(str).values
    gene_symbols = genes.iloc[:, 0].astype(str).values

gene_symbols = make_unique(gene_symbols)

# -------------------------------
# create one h5ad per child GSM
# -------------------------------
made_files = []
missing = []

for gsm in child_gsms:
    # find matrix and barcode files for this GSM
    matrix_path = find_one_file([
        os.path.join(raw_dir, f"{gsm}_*_matrix.mtx.gz"),
        os.path.join(raw_dir, f"{gsm}_*_matrix.mtx")
    ])

    barcodes_path = find_one_file([
        os.path.join(raw_dir, f"{gsm}_*_barcodes.tsv.gz"),
        os.path.join(raw_dir, f"{gsm}_*_barcodes.tsv")
    ])

    if matrix_path is None or barcodes_path is None:
        missing.append((gsm, matrix_path, barcodes_path))
        print(f"Skipping {gsm}: missing file(s)")
        continue

    # sample name = filename stem before "_matrix"
    # example: GSM4029907_JB17010
    base = os.path.basename(matrix_path)
    sample_name = base.replace("_matrix.mtx.gz", "").replace("_matrix.mtx", "")

    print(f"Processing {sample_name}...")

    # read sparse matrix
    X = mmread(matrix_path).tocsr()

    # GEO matrices are usually genes x cells -> transpose to cells x genes
    # we will confirm against barcode and gene lengths
    barcodes = read_table_auto(barcodes_path, header=None).iloc[:, 0].astype(str).values

    if X.shape[0] == len(gene_symbols) and X.shape[1] == len(barcodes):
        X = X.T   # convert to cells x genes
    elif X.shape[0] == len(barcodes) and X.shape[1] == len(gene_symbols):
        pass      # already cells x genes
    else:
        raise ValueError(
            f"{sample_name}: matrix shape {X.shape} does not match "
            f"{len(gene_symbols)} genes and {len(barcodes)} barcodes"
        )

    # build AnnData
    adata = sc.AnnData(X=X)

    # unique cell names: sample_barcode
    adata.obs_names = [f"{sample_name}_{bc}" for bc in barcodes]
    adata.var_names = gene_symbols

    # store extra gene/sample metadata
    adata.var["gene_ids"] = gene_ids
    adata.var["gene_symbols"] = adata.var_names
    adata.obs["gsm_id"] = gsm
    adata.obs["sample"] = sample_name

    # save
    out_path = os.path.join(out_dir, f"{sample_name}.h5ad")
    adata.write_h5ad(out_path)
    made_files.append(out_path)

print("\nDone.")
print(f"Created {len(made_files)} h5ad files.")
if missing:
    print("\nMissing files for these GSMs:")
    for item in missing:
        print(item)

**Per-sample QC filtering.** Cells are filtered to `min_genes=200`, genes to `min_cells=3`, and a mitochondrial-fraction cutoff of `pct_counts_mt < 25` is applied per sample before combining. 25% is a deliberately permissive ("very mild") threshold rather than the more common 10-20% used in cleaner PBMC datasets, chosen because this cohort spans many separately-processed patient samples with variable background mitochondrial content; a stricter global cutoff risked disproportionately removing otherwise-healthy cells from a subset of samples. This filter is applied identically in the adult pipeline (1D) for consistency.

In [ ]:
# Sensitivity of cell retention to reasonable QC threshold choices.
from glob import glob
from qc_utils import export_qc_threshold_sensitivity

child_qc_sensitivity = export_qc_threshold_sensitivity(
    glob(f"{BASE_DIR}/child_individual_h5ad/GSM*.h5ad"),
    f"{BASE_DIR}/Results/qc/child_qc_threshold_sensitivity.csv",
)
display(child_qc_sensitivity.head())


In [ ]:
#FILTER CELLS AND GENES TO MAKE INDIVIDUAL FILES BEFORE COMBINING -- RAM

import os
import glob
import scanpy as sc
from analysis_config import MIN_GENES_PER_CELL, MAX_MITOCHONDRIAL_PERCENT

in_dir = f"{BASE_DIR}/child_individual_h5ad"
out_dir = f"{BASE_DIR}/child_individual_h5ad_filtered"
os.makedirs(out_dir, exist_ok=True)

h5ad_files = sorted(glob.glob(os.path.join(in_dir, "GSM*.h5ad")))
print(f"Found {len(h5ad_files)} files")

for f in h5ad_files:
    sample_name = os.path.basename(f)
    print(f"\nProcessing {sample_name}")

    adata = sc.read_h5ad(f)
    print("  original shape:", adata.shape)

    # basic per-cell QC metric
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

    # filter cells and genes
    sc.pp.filter_cells(adata, min_genes=MIN_GENES_PER_CELL)
    sc.pp.filter_genes(adata, min_cells=3)

    # very mild mitochondrial filtering
    adata = adata[adata.obs["pct_counts_mt"] < MAX_MITOCHONDRIAL_PERCENT].copy()

    print("  filtered shape:", adata.shape)

    out_path = os.path.join(out_dir, sample_name)
    adata.write_h5ad(out_path)
    print("  saved:", out_path)

In [ ]:
# SCRUBLET — CHILD PER-SAMPLE DOUBLET FILTERING

import scanpy as sc
import scrublet as scr
from analysis_config import EXPECTED_DOUBLET_RATE
import numpy as np
from scipy import sparse
import os

input_dir = f"{BASE_DIR}/child_individual_h5ad_filtered"
output_dir = f"{BASE_DIR}/child_individual_h5ad_filtered_nodoublet"

os.makedirs(output_dir, exist_ok=True)
doublet_qc_records = []

for file in sorted(os.listdir(input_dir)):
    if file.startswith("GSM") and file.endswith(".h5ad"):

        input_path = os.path.join(input_dir, file)
        output_path = os.path.join(output_dir, file)

        print("\n==============================")
        print(f"Processing: {file}")

        adata = sc.read_h5ad(input_path)
        print("Original shape:", adata.shape)

        if np.issubdtype(adata.X.dtype, np.floating):
            print("⚠️ WARNING: Data may already be normalized/logged")

        counts_matrix = adata.X
        if sparse.issparse(counts_matrix):
            counts_matrix = counts_matrix.tocsc()
        else:
            counts_matrix = np.array(counts_matrix)

        scrub = scr.Scrublet(
            counts_matrix,
            expected_doublet_rate=EXPECTED_DOUBLET_RATE,
            random_state=0
        )

        doublet_scores, predicted_doublets = scrub.scrub_doublets(
            min_counts=2,
            min_cells=3,
            min_gene_variability_pctl=85,
            n_prin_comps=30
        )

        adata.obs["doublet_score"] = doublet_scores
        adata.obs["predicted_doublet"] = predicted_doublets.astype(bool)

        print("Doublet counts:")
        print(adata.obs["predicted_doublet"].value_counts())
        print("Doublet fraction:", adata.obs["predicted_doublet"].mean())

        adata_clean = adata[~adata.obs["predicted_doublet"]].copy()
        print("After removal:", adata_clean.shape)
        doublet_qc_records.append({
            "sample_file": file, "n_cells_before_doublet_filter": adata.n_obs,
            "n_predicted_doublets": int(predicted_doublets.sum()),
            "predicted_doublet_fraction": float(predicted_doublets.mean()),
            "n_cells_after_doublet_filter": adata_clean.n_obs,
        })

        adata_clean.write(output_path)
        print("Saved:", output_path)

from pathlib import Path
import pandas as pd
doublet_qc_path = Path(BASE_DIR) / "Results" / "qc" / "child_doublet_qc.csv"
doublet_qc_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(doublet_qc_records).to_csv(doublet_qc_path, index=False)

print("\nDONE: child per-sample doublet filtering complete.")

**Gene universe shrinkage from per-sample filtering.** Each sample's `min_cells=3` gene filter (previous step) keeps a slightly different gene set depending on that sample's own expression profile. Combining samples via `ad.concat(..., join="inner")` below keeps only genes present in *every* sample after that per-sample filter, so the final gene universe (32,738 -> 12,865 genes) is smaller than a naive union and is somewhat order-dependent (later-added samples can only ever narrow the intersection further, never widen it). This is a standard tradeoff for combining many independently-QC'd samples, not a bug, but worth stating explicitly since the final gene count is a real constraint on which ligand/receptor genes are available for the downstream LIANA analysis.

In [ ]:
# COMBINE IN H5AD IN CHUNKS (per-sample doublets already removed)

import os
import glob
import scanpy as sc
import anndata as ad

in_dir = f"{BASE_DIR}/child_individual_h5ad_filtered_nodoublet"
chunk_dir = f"{BASE_DIR}/child_chunks_nodoublet"
os.makedirs(chunk_dir, exist_ok=True)

files = sorted(glob.glob(os.path.join(in_dir, "*.h5ad")))

chunk_size = 5

for i in range(0, len(files), chunk_size):
    chunk_files = files[i:i+chunk_size]
    print(f"\nProcessing chunk {i//chunk_size + 1}")

    adatas = [sc.read_h5ad(f) for f in chunk_files]

    adata_chunk = ad.concat(
        adatas,
        axis=0,
        join="inner",
        merge="same",
        index_unique=None
    )

    out_path = os.path.join(chunk_dir, f"chunk_{i//chunk_size}.h5ad")
    adata_chunk.write_h5ad(out_path)

    print("Saved:", out_path, adata_chunk.shape)

    del adatas, adata_chunk

In [ ]:
import os
import glob
import gc
import scanpy as sc
import anndata as ad

chunk_dir = f"{BASE_DIR}/child_chunks_nodoublet"
superchunk_dir = f"{BASE_DIR}/child_superchunks"
os.makedirs(superchunk_dir, exist_ok=True)

chunk_files = sorted(glob.glob(os.path.join(chunk_dir, "*.h5ad")))
print(f"Found {len(chunk_files)} chunk files")

group_size = 2   # safer than 3 if RAM is tight

for group_idx, start in enumerate(range(0, len(chunk_files), group_size)):
    group = chunk_files[start:start + group_size]
    out_path = os.path.join(superchunk_dir, f"superchunk_{group_idx:02d}.h5ad")

    print(f"\nCreating {os.path.basename(out_path)} from:")
    for f in group:
        print("  ", os.path.basename(f))

    adata_merged = sc.read_h5ad(group[0])

    for f in group[1:]:
        adata_next = sc.read_h5ad(f)

        adata_merged = ad.concat(
            [adata_merged, adata_next],
            axis=0,
            join="inner",
            merge="same",
            index_unique=None
        )
        adata_merged.obs_names_make_unique()

        del adata_next
        gc.collect()

    print("Superchunk shape:", adata_merged.shape)
    adata_merged.write_h5ad(out_path)
    print("Saved:", out_path)

    del adata_merged
    gc.collect()

print("\nDone creating superchunks.")
print(sorted(glob.glob(os.path.join(superchunk_dir, "*.h5ad"))))

In [ ]:
# COMBINE SUPER CHUNKS

import os
import glob
import gc
import scanpy as sc
import anndata as ad

superchunk_dir = f"{BASE_DIR}/child_superchunks"
final_path = f"{BASE_DIR}/adata_child_combined.h5ad"

superchunk_files = sorted(glob.glob(os.path.join(superchunk_dir, "*.h5ad")))
print(f"Found {len(superchunk_files)} superchunk files")

if len(superchunk_files) == 0:
    raise ValueError("No superchunk files found.")

adata_child = sc.read_h5ad(superchunk_files[0])
print("Loaded first superchunk:", os.path.basename(superchunk_files[0]), adata_child.shape)

for f in superchunk_files[1:]:
    print("Adding superchunk:", os.path.basename(f))
    adata_next = sc.read_h5ad(f)

    adata_child = ad.concat(
        [adata_child, adata_next],
        axis=0,
        join="inner",
        merge="same",
        index_unique=None
    )
    adata_child.obs_names_make_unique()

    print("Current shape:", adata_child.shape)

    del adata_next
    gc.collect()

adata_child.write_h5ad(final_path)
print("Saved final child file:", final_path)
print(adata_child)

In [ ]:
# SANITY CHECK

import scanpy as sc
import pandas as pd
from scipy import sparse

path = f"{BASE_DIR}/adata_child_combined.h5ad"

adata = sc.read_h5ad(path)
print("Loaded:", adata)
print("Shape:", adata.shape)

print("\n--- BASIC STRUCTURE ---")
print("obs columns:", list(adata.obs.columns))
print("var columns:", list(adata.var.columns))
print("uns keys:", list(adata.uns.keys()))
print("obsm keys:", list(adata.obsm.keys()))
print("layers keys:", list(adata.layers.keys()))

print("\n--- MATRIX TYPE ---")
print("X type:", type(adata.X))
print("Is sparse:", sparse.issparse(adata.X))
print("dtype:", adata.X.dtype)

print("\n--- OBS NAMES ---")
print("obs unique:", adata.obs_names.is_unique)
print("first 5 obs_names:", list(adata.obs_names[:5]))

print("\n--- VAR NAMES ---")
print("var unique:", adata.var_names.is_unique)
print("first 20 var_names:", list(adata.var_names[:20]))
print("ENSG-like genes among first 100 var_names:",
      sum(str(x).startswith("ENSG") for x in adata.var_names[:100]))

print("\n--- SAMPLE INFO ---")
if "sample" in adata.obs.columns:
    print("number of samples:", adata.obs["sample"].nunique())
    print(adata.obs["sample"].value_counts().head(10))
else:
    print("WARNING: 'sample' column missing")

if "gsm_id" in adata.obs.columns:
    print("\nnumber of GSM IDs:", adata.obs["gsm_id"].nunique())
    print(adata.obs["gsm_id"].value_counts().head(10))
else:
    print("WARNING: 'gsm_id' column missing")

print("\n--- MISSING VALUES ---")
print("obs missing total:", int(adata.obs.isna().sum().sum()))
print("var missing total:", int(adata.var.isna().sum().sum()))

print("\n--- DUPLICATE / ZERO CHECKS ---")
if sparse.issparse(adata.X):
    row_sums = adata.X.sum(axis=1)
    col_sums = adata.X.sum(axis=0)
    row_sums = row_sums.A1 if hasattr(row_sums, "A1") else row_sums
    col_sums = col_sums.A1 if hasattr(col_sums, "A1") else col_sums
else:
    row_sums = adata.X.sum(axis=1)
    col_sums = adata.X.sum(axis=0)

print("cells with zero total counts:", int((row_sums == 0).sum()))
print("genes with zero total counts:", int((col_sums == 0).sum()))

print("\n--- OPTIONAL QC SUMMARY ---")
if "pct_counts_mt" in adata.obs.columns:
    print(adata.obs["pct_counts_mt"].describe())
else:
    print("pct_counts_mt not present")

if "n_genes_by_counts" in adata.obs.columns:
    print(adata.obs["n_genes_by_counts"].describe())
else:
    print("n_genes_by_counts not present")

print("\n--- QUICK SANITY FLAGS ---")
flags = []

if not adata.obs_names.is_unique:
    flags.append("obs_names are not unique")
if not adata.var_names.is_unique:
    flags.append("var_names are not unique")
if "sample" not in adata.obs.columns:
    flags.append("missing sample column")
if "gsm_id" not in adata.obs.columns:
    flags.append("missing gsm_id column")
if (row_sums == 0).sum() > 0:
    flags.append("there are cells with zero counts")
if (col_sums == 0).sum() > 0:
    flags.append("there are genes with zero counts")

if len(flags) == 0:
    print("No obvious structural problems found.")
else:
    print("Warnings:")
    for f in flags:
        print("-", f)

In [ ]:
import scanpy as sc
from scipy import sparse

in_path = f"{BASE_DIR}/adata_child_combined.h5ad"
norm_path = f"{BASE_DIR}/adata_child_combined_norm.h5ad"

adata = sc.read_h5ad(in_path)
print("Loaded:", adata)

# reduce memory footprint
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)

adata.X = adata.X.astype("float32")

# preserve unnormalized counts for reproducibility
adata.layers["counts"] = adata.X.copy()

# normalize
sc.pp.normalize_total(adata, target_sum=1e4)

print("Normalized:", adata)

adata.write_h5ad(norm_path)
print("Saved to:", norm_path)

In [ ]:
import scanpy as sc

in_path = f"{BASE_DIR}/adata_child_combined_norm.h5ad"
log_path = f"{BASE_DIR}/adata_child_combined_log1p.h5ad"

adata = sc.read_h5ad(in_path)
print("Loaded:", adata)

sc.pp.log1p(adata, chunked=True, chunk_size=2000)

adata.raw = adata
print("Log1p done:", adata)

adata.write_h5ad(log_path)
print("Saved to:", log_path)

In [ ]:
import scanpy as sc
from analysis_config import N_HIGHLY_VARIABLE_GENES

in_path = f"{BASE_DIR}/adata_child_combined_log1p.h5ad"
hvg_path = f"{BASE_DIR}/adata_child_combined_log1p_hvg.h5ad"

adata = sc.read_h5ad(in_path)
print("Loaded:", adata)

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_HIGHLY_VARIABLE_GENES,
    flavor="seurat"
)

print("Highly variable genes:", int(adata.var["highly_variable"].sum()))
adata.write_h5ad(hvg_path)
print("Saved:", hvg_path)

**Clustering resolution and HVG selection.** Three Leiden resolutions (0.2, 0.5, 0.8) are computed for comparison; **0.8** was selected as the final resolution used downstream (1B) because it was the finest resolution that still produced clusters mapping cleanly onto recognizable PBMC cell types via marker genes, without fragmenting any single cell type across multiple clusters. Lower resolutions under-split biologically distinct populations (e.g. merged T-cell subsets); the same resolution is used in the adult pipeline (1D) for consistency across cohorts.

Note also that highly-variable-gene selection (previous step, `sc.pp.highly_variable_genes`) is **not** batch-aware (no `batch_key="sample"`), so sample-to-sample technical variation could in principle be selected into the HVG set ahead of true biological variation, before Harmony ever runs. Harmony's batch correction on the resulting PCA embedding substantially mitigates this in practice (see the before/after UMAP-by-sample comparison in `Notebook_Outputs/batch_check_child_umap_by_sample*.png`), but a batch-aware HVG selection would be a more rigorous alternative if this pipeline is revisited.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(
    f"{BASE_DIR}/adata_child_combined_log1p_hvg.h5ad"
)

print("Loaded:", adata)
print("HVGs:", int(adata.var["highly_variable"].sum()))

# PCA directly on HVGs, no scaling
sc.tl.pca(adata, svd_solver="arpack", use_highly_variable=True, random_state=0)

# Batch correction across donors/samples (Harmony).
# harmonypy 0.0.10 provides an OS-independent wheel; Scanpy's wrapper handles
# that release's corrected-PC orientation and stores cells x components.
sc.external.pp.harmony_integrate(
    adata,
    key="sample",
    basis="X_pca",
    adjusted_basis="X_pca_harmony",
    random_state=0,
)
if adata.obsm["X_pca_harmony"].shape != adata.obsm["X_pca"].shape:
    raise ValueError("Harmony returned an unexpected corrected-PC shape.")

# neighbors + UMAP (on the batch-corrected embedding)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca_harmony", random_state=0)
sc.tl.umap(adata, random_state=0)

# Leiden at multiple resolutions
resolutions = [0.2, 0.5, 0.8]
for res in resolutions:
    key = f"leiden_res_{res}"
    sc.tl.leiden(adata, resolution=res, key_added=key, random_state=0)
    print(key, "->", adata.obs[key].nunique(), "clusters")

# save
out_path = f"{BASE_DIR}/child_individual_h5ad/adata_child_processed_multi_leiden.h5ad"
adata.write_h5ad(out_path)
print("Saved:", out_path)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_res_0.2", "leiden_res_0.5", "leiden_res_0.8"],
    wspace=0.4
, save="_figure_01.png")

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_res_0.8"],
    wspace=0.4
, save="_figure_02.png")

In [ ]:
adata.write_h5ad(f"{BASE_DIR}/child_individual_h5ad/adata_child_processed_selected_leiden.h5ad")

In [ ]:
adata.obs["leiden"] = adata.obs["leiden_res_0.8"]

In [ ]:
adata.write_h5ad(f"{BASE_DIR}/child_individual_h5ad/adata_child_processed_leiden08.h5ad")

In [ ]:
# Publication QC summary from the retained post-filter object.
from qc_utils import export_sample_qc

qc_table = export_sample_qc(adata, f"{BASE_DIR}/Results/qc/child_postfilter_sample_qc.csv")
display(qc_table)


In [ ]:
# Remove only checkpoints that no downstream notebook consumes.
from publication_utils import remove_owned_intermediates

remove_owned_intermediates(BASE_DIR, [
    "child_individual_h5ad_filtered",
    "child_individual_h5ad_filtered_nodoublet",
    "child_chunks_nodoublet",
    "child_superchunks",
    "adata_child_combined.h5ad",
    "adata_child_combined_norm.h5ad",
    "adata_child_combined_log1p.h5ad"
])
print("Removed disposable preprocessing checkpoints.")
